# مسئلهٔ ۱ — V2 / مدل A4: ResNet18 + GRU

هر sequence شامل featureهای مرتب `[16, 512]` از ResNet18 است. GRU این دنباله را به‌ترتیب زمانی می‌خواند و hidden state نهایی را برای طبقه‌بندی ویدئویی استفاده می‌کند.

این آزمایش با GRU نسخهٔ V1 متفاوت است: اکنون فریم‌ها نزدیک رخداد، مرتب، یک‌sequence-per-video و بدون label نویز فریم‌به‌فریم‌اند. ResNet18 freeze می‌ماند تا اثر GRU به‌تنهایی با A1–A3 مقایسه شود.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import random

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

DATA_ROOT = Path(r'P:\NexarCollisionData')
SEQUENCE_MANIFEST_PATH = DATA_ROOT / 'sequence_manifest_v2.csv'
FEATURE_CACHE_PATH = DATA_ROOT / 'processed_v2' / 'resnet18_imagenet_features_v2_w2_16x224x320.pt'
MODEL_DIR = DATA_ROOT / 'models_v2'
MODEL_NAME = 'resnet18_gru_frozen'

NUM_FRAMES = 16
FEATURE_DIM = 512
GRU_HIDDEN_DIM = 256
BATCH_SIZE = 64
EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0
SEED = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert SEQUENCE_MANIFEST_PATH.exists(), 'Run notebook 08 first.'
assert FEATURE_CACHE_PATH.exists(), 'Run notebook 10 first to create the feature cache.'
print(f'Device: {device}')

Device: cpu


In [2]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
sequence_manifest = pd.read_csv(SEQUENCE_MANIFEST_PATH).copy()
sequence_manifest['video_id'] = sequence_manifest['video_id'].astype(str)
sequence_manifest['label'] = sequence_manifest['label'].astype(int)
sequence_manifest = sequence_manifest.sort_values('video_id', key=lambda series: series.astype(int)).reset_index(drop=True)

feature_payload = torch.load(FEATURE_CACHE_PATH, map_location='cpu', weights_only=False)
features = feature_payload['features'].float()
labels = feature_payload['labels'].long()
assert features.shape == (600, NUM_FRAMES, FEATURE_DIM)
assert labels.tolist() == sequence_manifest['label'].tolist()
assert feature_payload['sequence_ids'] == sequence_manifest['sequence_id'].tolist()

train_indices = np.flatnonzero(sequence_manifest['split'].eq('train').to_numpy())
validation_indices = np.flatnonzero(sequence_manifest['split'].eq('validation').to_numpy())
assert len(train_indices) == 480 and len(validation_indices) == 120
display(pd.crosstab(sequence_manifest['split'], sequence_manifest['label']))

label,0,1
split,,
train,240,240
validation,60,60


In [3]:
class SequenceFeatureDataset(Dataset):
    def __init__(self, features: torch.Tensor, labels: torch.Tensor, indices: np.ndarray):
        self.features = features
        self.labels = labels
        self.indices = torch.as_tensor(indices, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, index: int):
        source_index = self.indices[index]
        return self.features[source_index], self.labels[source_index], int(source_index)

class ResNet18GRUClassifier(nn.Module):
    def __init__(self, feature_dim: int = FEATURE_DIM, hidden_dim: int = GRU_HIDDEN_DIM, dropout: float = 0.35):
        super().__init__()
        self.gru = nn.GRU(feature_dim, hidden_dim, num_layers=1, batch_first=True)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, sequence_features: torch.Tensor, frame_mask: torch.Tensor | None = None) -> torch.Tensor:
        # All V2 cached sequences are complete; frame_mask is retained for later robustness.
        if frame_mask is not None and not frame_mask.bool().all():
            lengths = frame_mask.long().sum(dim=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(sequence_features, lengths, batch_first=True, enforce_sorted=False)
            _, hidden = self.gru(packed)
        else:
            _, hidden = self.gru(sequence_features)
        return self.classifier(hidden[-1]).squeeze(1)

train_loader = DataLoader(SequenceFeatureDataset(features, labels, train_indices), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(SequenceFeatureDataset(features, labels, validation_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
model = ResNet18GRUClassifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()
print(model)

ResNet18GRUClassifier(
  (gru): GRU(512, 256, batch_first=True)
  (classifier): Sequential(
    (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    (1): Dropout(p=0.35, inplace=False)
    (2): Linear(in_features=256, out_features=1, bias=True)
  )
)


In [4]:
def binary_metrics(y_true: np.ndarray, probabilities: np.ndarray, threshold: float) -> dict:
    predictions = (probabilities >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, predictions)),
        'precision': float(precision_score(y_true, predictions, zero_division=0)),
        'recall': float(recall_score(y_true, predictions, zero_division=0)),
        'f1': float(f1_score(y_true, predictions, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, probabilities)),
        'pr_auc': float(average_precision_score(y_true, probabilities)),
        'confusion_matrix': confusion_matrix(y_true, predictions).tolist(),
    }

def evaluate(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    labels_out, probabilities_out, indices_out = [], [], []
    with torch.inference_mode():
        for batch_features, batch_labels, batch_indices in loader:
            logits = model(batch_features.to(device))
            labels_out.append(batch_labels.numpy())
            probabilities_out.append(torch.sigmoid(logits).cpu().numpy())
            indices_out.append(batch_indices.numpy())
    return np.concatenate(labels_out), np.concatenate(probabilities_out), np.concatenate(indices_out)

best_pr_auc = -np.inf
best_epoch = 0
epochs_without_improvement = 0
history = []
best_model_path = MODEL_DIR / f'{MODEL_NAME}_best.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum = 0.0
    for batch_features, batch_labels, _ in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.float().to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(batch_features), batch_labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        optimizer.step()
        loss_sum += loss.item() * len(batch_labels)

    validation_labels, validation_probabilities, _ = evaluate(model, validation_loader)
    metrics_at_05 = binary_metrics(validation_labels, validation_probabilities, 0.5)
    record = {
        'epoch': epoch,
        'train_loss': loss_sum / len(train_loader.dataset),
        'validation_accuracy_at_0_5': metrics_at_05['accuracy'],
        'validation_f1_at_0_5': metrics_at_05['f1'],
        'validation_recall_at_0_5': metrics_at_05['recall'],
        'validation_pr_auc': metrics_at_05['pr_auc'],
        'validation_roc_auc': metrics_at_05['roc_auc'],
    }
    history.append(record)
    print(record)

    if record['validation_pr_auc'] > best_pr_auc:
        best_pr_auc = record['validation_pr_auc']
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            'model_state_dict': model.state_dict(), 'epoch': epoch,
            'validation_pr_auc': best_pr_auc, 'model_name': MODEL_NAME,
            'feature_dim': FEATURE_DIM, 'gru_hidden_dim': GRU_HIDDEN_DIM,
            'num_frames': NUM_FRAMES, 'encoder': 'ResNet18_Weights.IMAGENET1K_V1 (frozen)',
        }, best_model_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping at epoch {epoch}; best epoch: {best_epoch}')
            break

history_path = MODEL_DIR / f'{MODEL_NAME}_training_history.csv'
pd.DataFrame(history).to_csv(history_path, index=False)
print(f'Best epoch by validation PR-AUC: {best_epoch}, PR-AUC={best_pr_auc:.4f}')

{'epoch': 1, 'train_loss': 0.811624530951182, 'validation_accuracy_at_0_5': 0.5583333333333333, 'validation_f1_at_0_5': 0.6900584795321637, 'validation_recall_at_0_5': 0.9833333333333333, 'validation_pr_auc': 0.735360584730939, 'validation_roc_auc': 0.7413888888888888}
{'epoch': 2, 'train_loss': 0.6360068519910177, 'validation_accuracy_at_0_5': 0.7083333333333334, 'validation_f1_at_0_5': 0.7107438016528925, 'validation_recall_at_0_5': 0.7166666666666667, 'validation_pr_auc': 0.7296006296403867, 'validation_roc_auc': 0.7536111111111111}
{'epoch': 3, 'train_loss': 0.581568201382955, 'validation_accuracy_at_0_5': 0.7416666666666667, 'validation_f1_at_0_5': 0.7350427350427351, 'validation_recall_at_0_5': 0.7166666666666667, 'validation_pr_auc': 0.732729916132892, 'validation_roc_auc': 0.7613888888888888}
{'epoch': 4, 'train_loss': 0.5588876366615295, 'validation_accuracy_at_0_5': 0.7, 'validation_f1_at_0_5': 0.6666666666666666, 'validation_recall_at_0_5': 0.6, 'validation_pr_auc': 0.734054

In [5]:
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
validation_labels, validation_probabilities, validation_indices_out = evaluate(model, validation_loader)

threshold_table = pd.DataFrame([
    binary_metrics(validation_labels, validation_probabilities, float(threshold))
    for threshold in np.round(np.arange(0.10, 0.901, 0.01), 2)
])
selected_row = threshold_table.sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0]
selected_threshold = float(selected_row['threshold'])
metrics_at_05 = binary_metrics(validation_labels, validation_probabilities, 0.5)
metrics_at_selected_threshold = binary_metrics(validation_labels, validation_probabilities, selected_threshold)

prediction_table = sequence_manifest.iloc[validation_indices_out].copy().reset_index(drop=True)
prediction_table['positive_probability'] = validation_probabilities
prediction_table['prediction_at_0_5'] = (validation_probabilities >= 0.5).astype(int)
prediction_table['prediction_at_selected_threshold'] = (validation_probabilities >= selected_threshold).astype(int)
prediction_table['selected_threshold'] = selected_threshold

predictions_path = MODEL_DIR / f'{MODEL_NAME}_validation_predictions.csv'
threshold_path = MODEL_DIR / f'{MODEL_NAME}_threshold_curve.csv'
metrics_path = MODEL_DIR / f'{MODEL_NAME}_metrics.json'
prediction_table.to_csv(predictions_path, index=False)
threshold_table.to_csv(threshold_path, index=False)

metrics_payload = {
    'model_name': MODEL_NAME,
    'evaluation_scope': 'clip-level validation on fixed V2-W2 sequences; not full-MP4 sliding-window inference',
    'selection_metric': 'validation PR-AUC at the saved checkpoint',
    'best_epoch': int(checkpoint['epoch']),
    'selected_threshold_by_validation_f1': selected_threshold,
    'metrics_at_threshold_0_5': metrics_at_05,
    'metrics_at_selected_threshold': metrics_at_selected_threshold,
    'encoder': 'ResNet18 ImageNet frozen',
    'temporal_aggregator': 'unidirectional GRU final hidden state over 16 frame features',
    'input_shape_per_sequence': [NUM_FRAMES, 3, 224, 320],
}
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

figure, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(validation_labels, (validation_probabilities >= 0.5).astype(int), ax=axes[0], colorbar=False)
axes[0].set_title('Validation, threshold = 0.50')
ConfusionMatrixDisplay.from_predictions(validation_labels, (validation_probabilities >= selected_threshold).astype(int), ax=axes[1], colorbar=False)
axes[1].set_title(f'Validation, threshold = {selected_threshold:.2f}')
figure.tight_layout()
confusion_path = MODEL_DIR / f'{MODEL_NAME}_confusion_matrices.png'
figure.savefig(confusion_path, dpi=160)
plt.close(figure)

comparison_sources = [
    ('A1 ResNet18 + mean pooling', MODEL_DIR / 'resnet18_mean_pooling_frozen_metrics.json'),
    ('A2 ResNet18 + mean-max pooling', MODEL_DIR / 'resnet18_meanmax_pooling_frozen_metrics.json'),
    ('A3 ResNet18 + temporal attention', MODEL_DIR / 'resnet18_temporal_attention_frozen_metrics.json'),
]
comparison_rows = []
for name, path in comparison_sources:
    if path.exists():
        payload = json.loads(path.read_text(encoding='utf-8'))
        result = payload['metrics_at_selected_threshold']
        comparison_rows.append({
            'model': name, 'f1': result['f1'], 'recall': result['recall'],
            'precision': result['precision'], 'pr_auc': result['pr_auc'],
            'threshold': payload['selected_threshold_by_validation_f1'],
        })
comparison_rows.append({
    'model': 'A4 ResNet18 + GRU',
    'f1': metrics_at_selected_threshold['f1'], 'recall': metrics_at_selected_threshold['recall'],
    'precision': metrics_at_selected_threshold['precision'], 'pr_auc': metrics_at_selected_threshold['pr_auc'],
    'threshold': selected_threshold,
})
comparison_table = pd.DataFrame(comparison_rows)
comparison_path = MODEL_DIR / 'v2_temporal_model_ablation_comparison.csv'
comparison_table.to_csv(comparison_path, index=False)

print('Metrics at threshold 0.50:')
print(metrics_at_05)
print('Metrics at validation-selected threshold:')
print(metrics_at_selected_threshold)
display(comparison_table)
print(f'Model: {best_model_path}')
print(f'Comparison: {comparison_path}')

Metrics at threshold 0.50:
{'threshold': 0.5, 'accuracy': 0.7, 'precision': 0.75, 'recall': 0.6, 'f1': 0.6666666666666666, 'roc_auc': 0.7805555555555556, 'pr_auc': 0.7613366687253724, 'confusion_matrix': [[48, 12], [24, 36]]}
Metrics at validation-selected threshold:
{'threshold': 0.17, 'accuracy': 0.7166666666666667, 'precision': 0.6666666666666666, 'recall': 0.8666666666666667, 'f1': 0.7536231884057971, 'roc_auc': 0.7805555555555556, 'pr_auc': 0.7613366687253724, 'confusion_matrix': [[34, 26], [8, 52]]}


,model,f1,recall,precision,pr_auc,threshold
0,A1 ResNet18 + mean pooling,0.762712,0.750000,0.775862,0.727511,0.47
1,A2 ResNet18 + mean-max pooling,0.787402,0.833333,0.746269,0.742437,0.44
2,A3 ResNet18 + temporal attention,0.773723,0.883333,0.688312,0.716780,0.39
3,A4 ResNet18 + GRU,0.753623,0.866667,0.666667,0.761337,0.17


Model: P:\NexarCollisionData\models_v2\resnet18_gru_frozen_best.pt
Comparison: P:\NexarCollisionData\models_v2\v2_temporal_model_ablation_comparison.csv


## تصمیم پس از A4

اگر GRU از A2 بهتر نشد، پیچیدگی زمانی به‌تنهایی ارزش افزوده‌ای نداشته و A2 محفوظ می‌ماند. اگر نزدیک یا بهتر بود، گام A5 یعنی GRU + temporal attention آزمایش می‌شود. BiLSTM + additive attention + FFN فقط پس از این مقایسه وارد می‌شود.